### Replace request body with realistic test case — use something resembling the AAM0658 malicious day pattern we found earlier:

{
  "logon_count": 2,
  "device_count": 6,
  "file_count": 10,
  "email_count": 5,
  "http_count": 100,
  "logon_count_deviation": 0,
  "device_count_deviation": 6,
  "file_count_deviation": 8,
  "email_count_deviation": 2,
  "http_count_deviation": 10,
  "after_hours_logon_count": 1,
  "external_email_count": 3,
  "is_first_usb_use": 1
}

### 3 sample predictions + 1 known failure case

In [1]:
# Run this in a notebook to find good examples from your real data
import pandas as pd

combined = pd.read_csv("../data/processed/isolation_forest_results.csv")  # or wherever your merged results are

# Sample 1: A correctly caught scenario day (true positive)
true_positive = combined[(combined['is_scenario_day']==True)].sort_values('anomaly_score').iloc[0]
print("TRUE POSITIVE EXAMPLE:")
print(true_positive[['user','day','is_scenario_day','anomaly_score']])

# Sample 2: A normal day correctly scored low (true negative)
true_negative = combined[(combined['is_scenario_day']==False)].sample(1, random_state=1).iloc[0]
print("\nTRUE NEGATIVE EXAMPLE:")
print(true_negative[['user','day','is_scenario_day','anomaly_score']])

# Sample 3: Another interesting case (maybe the RKD0604 unlabeled anomaly)
special_case = combined[(combined['user']=='RKD0604')]
print("\nSPECIAL CASE (unlabeled anomaly):")
print(special_case[['user','day','is_scenario_day','anomaly_score']].head())

# Known FAILURE CASE: a real scenario day that was MISSED (scored as normal)
missed = combined[(combined['is_scenario_day']==True)].sort_values('anomaly_score', ascending=False).iloc[0]
print("\nKNOWN FAILURE CASE (missed insider):")
print(missed[['user','day','is_scenario_day','anomaly_score']])

TRUE POSITIVE EXAMPLE:
user                  MCF0600
day                2010-09-20
is_scenario_day          True
anomaly_score       -0.166865
Name: 142234, dtype: object

TRUE NEGATIVE EXAMPLE:
user                  CAB0614
day                2010-10-22
is_scenario_day         False
anomaly_score        0.129286
Name: 33311, dtype: object

SPECIAL CASE (unlabeled anomaly):
           user         day  is_scenario_day  anomaly_score
182214  RKD0604  2010-06-01            False       0.222463
182215  RKD0604  2010-06-02            False       0.225882
182216  RKD0604  2010-06-03            False       0.181867
182217  RKD0604  2010-06-04            False       0.219741
182218  RKD0604  2010-06-07            False       0.225882

KNOWN FAILURE CASE (missed insider):
user                  KPC0073
day                2010-07-08
is_scenario_day          True
anomaly_score        0.232291
Name: 127796, dtype: object


In [2]:
final_features = pd.read_csv("../data/processed/final_features.csv")
final_features['day'] = pd.to_datetime(final_features['day'])

feature_cols = [
    'logon_count', 'device_count', 'file_count', 'email_count', 'http_count',
    'logon_count_deviation', 'device_count_deviation', 'file_count_deviation',
    'email_count_deviation', 'http_count_deviation',
    'after_hours_logon_count', 'external_email_count', 'is_first_usb_use'
]

cases = [
    ('MCF0600', '2010-09-20', 'True Positive'),
    ('CAB0614', '2010-10-22', 'True Negative'),
    ('RKD0604', '2010-07-09', 'Special Case - Unlabeled Anomaly'),
    ('KPC0073', '2010-07-08', 'Known Failure - Missed Insider')
]

for user, day, label in cases:
    row = final_features[(final_features['user']==user) & (final_features['day']==pd.Timestamp(day))]
    print(f"\n=== {label}: {user} on {day} ===")
    print(row[feature_cols].to_dict('records'))


=== True Positive: MCF0600 on 2010-09-20 ===
[{'logon_count': 7, 'device_count': 20.0, 'file_count': 27.0, 'email_count': 1.0, 'http_count': 12.0, 'logon_count_deviation': 3.857142857142857, 'device_count_deviation': 20.0, 'file_count_deviation': 27.0, 'email_count_deviation': 0.0, 'http_count_deviation': 2.1428571428571423, 'after_hours_logon_count': 3, 'external_email_count': 1.0, 'is_first_usb_use': True}]

=== True Negative: CAB0614 on 2010-10-22 ===
[{'logon_count': 3, 'device_count': 2.0, 'file_count': 7.0, 'email_count': 16.0, 'http_count': 162.0, 'logon_count_deviation': -0.7142857142857144, 'device_count_deviation': -0.2857142857142856, 'file_count_deviation': 2.428571428571429, 'email_count_deviation': 2.0, 'http_count_deviation': 0.0, 'after_hours_logon_count': 0, 'external_email_count': 3.0, 'is_first_usb_use': False}]

=== Special Case - Unlabeled Anomaly: RKD0604 on 2010-07-09 ===
[{'logon_count': 3, 'device_count': 18.0, 'file_count': 27.0, 'email_count': 15.0, 'http_co

## Deployment Demo — Sample Predictions

### 1. True Positive: User MCF0600, 2010-09-20 (confirmed scenario day)
**Input:** logon_count=7, device_count=20, file_count=27, first_usb_use=True, after_hours_logon_count=3

**API Response:**
```json
{
  "isolation_forest_score": -0.16686463826795328,
  "isolation_forest_flagged": true,
  "autoencoder_reconstruction_error": 35.82804377392311,
  "ensemble_indicator": 35.994908412191066,
  "risk_level": "HIGH"
}
```
Both models correctly flag this confirmed scenario day as high risk, driven by the first-time USB use and after-hours activity.

### 2. True Negative: User CAB0614, 2010-10-22 (normal day)
**Input:** logon_count=3, device_count=2, file_count=7, first_usb_use=False, after_hours_logon_count=0

**API Response:**
```json
{
  "isolation_forest_score": 0.1292862581739569,
  "isolation_forest_flagged": false,
  "autoencoder_reconstruction_error": 0.3521314941768232,
  "ensemble_indicator": 0.22284523600286632,
  "risk_level": "LOW"
}
```
Both models correctly clear this normal day as low risk.

### 3. Special Case: User RKD0604, 2010-07-09 (unlabeled anomaly, discussed in Day 10)
**Input:** logon_count=3, device_count=18, file_count=27, first_usb_use=True, after_hours_logon_count=1, external_email_count=5

**API Response:**
```json
{
  "isolation_forest_score": -0.11468990854929728,
  "isolation_forest_flagged": true,
  "autoencoder_reconstruction_error": 37.360431666230475,
  "ensemble_indicator": 37.47512157477977,
  "risk_level": "HIGH"
}
```
This user is confirmed malicious but this specific day falls outside their officially labeled scenario window. The model still correctly surfaces it as the single highest-risk case in the entire test set a first-time USB spike with correlated file activity, matching the same signature as the confirmed Scenario 1 pattern.

### 4. Known Failure Case: User KPC0073, 2010-07-08 (missed insider — scenario day scored as low-risk)
**Input:** logon_count=2, device_count=0, file_count=0, first_usb_use=False, after_hours_logon_count=0

**API Response:**
```json
{
  "isolation_forest_score": 0.23229149666985138,
  "isolation_forest_flagged": false,
  "autoencoder_reconstruction_error": 0.03366122291454252,
  "ensemble_indicator": -0.19863027375530887,
  "risk_level": "LOW"
}
```
Both models score this confirmed scenario day as low risk, missing it entirely.

**This demonstrates a real limitation:** not all injected scenario behavior produces a strong anomaly signal in the engineered features, particularly if the malicious activity that day doesn't deviate sharply from the user's personal baseline. In this case, zero device/file activity and no after-hours logons meant the model had no elevated signal to act on suggesting the malicious behavior that day likely manifested through a channel (e.g., email content, subtler access patterns) not captured by the current count-based features.

## Deployment Demo — Sample Predictions

### 1. True Positive: User MCF0600, 2010-09-20 (confirmed scenario day)
**Input:** logon_count=7, device_count=20, file_count=27, first_usb_use=True, after_hours_logon_count=3

**API Response:**
```json
{
  "isolation_forest_score": -0.16686463826795328,
  "isolation_forest_flagged": true,
  "autoencoder_reconstruction_error": 35.82804377392311,
  "ensemble_indicator": 35.994908412191066,
  "risk_level": "HIGH"
}
```
Both models correctly flag this confirmed scenario day as high risk, driven by the first-time USB use and after-hours activity.

### 2. True Negative: User CAB0614, 2010-10-22 (normal day)
**Input:** logon_count=3, device_count=2, file_count=7, first_usb_use=False, after_hours_logon_count=0

**API Response:**
```json
{
  "isolation_forest_score": 0.1292862581739569,
  "isolation_forest_flagged": false,
  "autoencoder_reconstruction_error": 0.3521314941768232,
  "ensemble_indicator": 0.22284523600286632,
  "risk_level": "LOW"
}
```
Both models correctly clear this normal day as low risk.

### 3. Special Case: User RKD0604, 2010-07-09 (unlabeled anomaly, discussed in Day 10)
**Input:** logon_count=3, device_count=18, file_count=27, first_usb_use=True, after_hours_logon_count=1, external_email_count=5

**API Response:**
```json
{
  "isolation_forest_score": -0.11468990854929728,
  "isolation_forest_flagged": true,
  "autoencoder_reconstruction_error": 37.360431666230475,
  "ensemble_indicator": 37.47512157477977,
  "risk_level": "HIGH"
}
```
This user is confirmed malicious but this specific day falls outside their officially labeled scenario window. The model still correctly surfaces it as the single highest-risk case in the entire test set a first-time USB spike with correlated file activity, matching the same signature as the confirmed Scenario 1 pattern.

### 4. Known Failure Case: User KPC0073, 2010-07-08 (missed insider — scenario day scored as low-risk)
**Input:** logon_count=2, device_count=0, file_count=0, first_usb_use=False, after_hours_logon_count=0

**API Response:**
```json
{
  "isolation_forest_score": 0.23229149666985138,
  "isolation_forest_flagged": false,
  "autoencoder_reconstruction_error": 0.03366122291454252,
  "ensemble_indicator": -0.19863027375530887,
  "risk_level": "LOW"
}
```
Both models score this confirmed scenario day as low risk, missing it entirely.

**This demonstrates a real limitation:** not all injected scenario behavior produces a strong anomaly signal in the engineered features, particularly if the malicious activity that day doesn't deviate sharply from the user's personal baseline. In this case, zero device/file activity and no after-hours logons meant the model had no elevated signal to act on suggesting the malicious behavior that day likely manifested through a channel (e.g., email content, subtler access patterns) not captured by the current count-based features.

## Deployment Demo — Sample Predictions

### 1. True Positive: User MCF0600, 2010-09-20 (confirmed scenario day)
**Input:** logon_count=7, device_count=20, file_count=27, first_usb_use=True, after_hours_logon_count=3

**API Response:**
```json
{
  "isolation_forest_score": -0.16686463826795328,
  "isolation_forest_flagged": true,
  "autoencoder_reconstruction_error": 35.82804377392311,
  "ensemble_indicator": 35.994908412191066,
  "risk_level": "HIGH"
}
```
Both models correctly flag this confirmed scenario day as high risk, driven by the first-time USB use and after-hours activity.

### 2. True Negative: User CAB0614, 2010-10-22 (normal day)
**Input:** logon_count=3, device_count=2, file_count=7, first_usb_use=False, after_hours_logon_count=0

**API Response:**
```json
{
  "isolation_forest_score": 0.1292862581739569,
  "isolation_forest_flagged": false,
  "autoencoder_reconstruction_error": 0.3521314941768232,
  "ensemble_indicator": 0.22284523600286632,
  "risk_level": "LOW"
}
```
Both models correctly clear this normal day as low risk.

### 3. Special Case: User RKD0604, 2010-07-09 (unlabeled anomaly, discussed in Day 10)
**Input:** logon_count=3, device_count=18, file_count=27, first_usb_use=True, after_hours_logon_count=1, external_email_count=5

**API Response:**
```json
{
  "isolation_forest_score": -0.11468990854929728,
  "isolation_forest_flagged": true,
  "autoencoder_reconstruction_error": 37.360431666230475,
  "ensemble_indicator": 37.47512157477977,
  "risk_level": "HIGH"
}
```
This user is confirmed malicious but this specific day falls outside their officially labeled scenario window. The model still correctly surfaces it as the single highest-risk case in the entire test set a first-time USB spike with correlated file activity, matching the same signature as the confirmed Scenario 1 pattern.

### 4. Known Failure Case: User KPC0073, 2010-07-08 (missed insider — scenario day scored as low-risk)
**Input:** logon_count=2, device_count=0, file_count=0, first_usb_use=False, after_hours_logon_count=0

**API Response:**
```json
{
  "isolation_forest_score": 0.23229149666985138,
  "isolation_forest_flagged": false,
  "autoencoder_reconstruction_error": 0.03366122291454252,
  "ensemble_indicator": -0.19863027375530887,
  "risk_level": "LOW"
}
```
Both models score this confirmed scenario day as low risk, missing it entirely.

**This demonstrates a real limitation:** not all injected scenario behavior produces a strong anomaly signal in the engineered features, particularly if the malicious activity that day doesn't deviate sharply from the user's personal baseline. In this case, zero device/file activity and no after-hours logons meant the model had no elevated signal to act on suggesting the malicious behavior that day likely manifested through a channel (e.g., email content, subtler access patterns) not captured by the current count-based features.

## Deployment Demo — Sample Predictions

The FastAPI scoring service (`deployment/app.py`) loads the trained 
Isolation Forest, StandardScaler, and Autoencoder, and returns a combined 
risk assessment for any user-day feature record.

| Case | User / Day | Iso. Forest Flag | Autoencoder Error | Risk Level | Ground Truth |
|---|---|---|---|---|---|
| True Positive | MCF0600, 2010-09-20 | Yes | 35.83 | HIGH | Confirmed scenario day |
| True Negative | CAB0614, 2010-10-22 | No | 0.35 | LOW | Normal day |
| Unlabeled Anomaly | RKD0604, 2010-07-09 | Yes | 37.36 | HIGH | Malicious user, outside labeled window (see Day 10 finding) |
| **Known Failure Case** | KPC0073, 2010-07-08 | No | 0.03 | **LOW** | **Confirmed scenario day — MISSED** |

### Known Limitation
User KPC0073's scenario-day activity on 2010-07-08 shows no elevated 
device, file, or after-hours signals (all near-zero deviations from 
baseline). This demonstrates that the current feature set does not 
capture every form of malicious behavior — likely because this 
particular day's insider activity manifested through channels (e.g., 
email content, subtler access patterns) not captured by the current 
count-based features. This is an honest, documented gap for future 
feature engineering work.

## Deployment Demo — Sample Predictions

The FastAPI scoring service (`deployment/app.py`) loads the trained 
Isolation Forest, StandardScaler, and Autoencoder, and returns a combined 
risk assessment for any user-day feature record.

| Case | User / Day | Iso. Forest Flag | Autoencoder Error | Risk Level | Ground Truth |
|---|---|---|---|---|---|
| True Positive | MCF0600, 2010-09-20 | Yes | 35.83 | HIGH | Confirmed scenario day |
| True Negative | CAB0614, 2010-10-22 | No | 0.35 | LOW | Normal day |
| Unlabeled Anomaly | RKD0604, 2010-07-09 | Yes | 37.36 | HIGH | Malicious user, outside labeled window (see Day 10 finding) |
| **Known Failure Case** | KPC0073, 2010-07-08 | No | 0.03 | **LOW** | **Confirmed scenario day — MISSED** |

### Known Limitation
User KPC0073's scenario-day activity on 2010-07-08 shows no elevated 
device, file, or after-hours signals (all near-zero deviations from 
baseline). This demonstrates that the current feature set does not 
capture every form of malicious behavior — likely because this 
particular day's insider activity manifested through channels (e.g., 
email content, subtler access patterns) not captured by the current 
count-based features. This is an honest, documented gap for future 
feature engineering work.